In [1]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 4.3 MB/s eta 0:00:00


In [2]:
from google.colab import files

archivos_subidos = files.upload()

Saving Politicas_Internas_NexaShip.pdf to Politicas_Internas_NexaShip.pdf


In [3]:
from pypdf import PdfReader

# Ruta del documento
ruta_pdf = "Politicas_Internas_NexaShip.pdf"

# Abrir el PDF
lector = PdfReader(ruta_pdf)

# Mostrar cantidad de páginas
print(f"Total de páginas encontradas: {len(lector.pages)}")

Total de páginas encontradas: 10


In [4]:
# Lista donde guardaremos el contenido de cada página
paginas_texto = []

for numero_pagina, pagina in enumerate(lector.pages, start=1):
    texto = pagina.extract_text()

    paginas_texto.append({
        "pagina": numero_pagina,
        "texto": texto
    })

print(f"Páginas procesadas correctamente: {len(paginas_texto)}")

Páginas procesadas correctamente: 10


In [5]:
# Mostrar una muestra del texto extraído de la página 3
print("Página:", paginas_texto[2]["pagina"])
print("-" * 50)
print(paginas_texto[2]["texto"][:1500])

Página: 3
--------------------------------------------------
NexaShip
NX-POL-001 | Versión 1.0 | Uso Interno
Políticas Internas de NexaShip
Página 3
1. OBJETIVO Y ALCANCE
1.1 Objetivo
Establecer lineamientos claros para que los equipos de NexaShip gestionen oportunidades comerciales, solicitudes de
prospectos y clientes, negociaciones, información corporativa y consultas internas de forma consistente, trazable y
alineada con las capacidades oficialmente documentadas de la empresa.
Estas políticas buscan reducir riesgos derivados de compromisos no autorizados, respuestas basadas en supuestos,
manejo inadecuado de información y escalamiento incorrecto de consultas.
1.2 Alcance
Las disposiciones de este documento aplican a:
 Equipo Comercial.
 
 Ejecutivos de cuenta.
 
 Account Managers.
 
 Soporte Comercial.
 
 Líderes de equipos comerciales.
 
 Gerencia Comercial.
 
 Personal que participe en validaciones de oportunidades.
 
 Áreas especializadas que intervengan en procesos de e

In [6]:
print("Página:", paginas_texto[2]["pagina"])
print("-" * 50)
print(paginas_texto[2]["texto"])

Página: 3
--------------------------------------------------
NexaShip
NX-POL-001 | Versión 1.0 | Uso Interno
Políticas Internas de NexaShip
Página 3
1. OBJETIVO Y ALCANCE
1.1 Objetivo
Establecer lineamientos claros para que los equipos de NexaShip gestionen oportunidades comerciales, solicitudes de
prospectos y clientes, negociaciones, información corporativa y consultas internas de forma consistente, trazable y
alineada con las capacidades oficialmente documentadas de la empresa.
Estas políticas buscan reducir riesgos derivados de compromisos no autorizados, respuestas basadas en supuestos,
manejo inadecuado de información y escalamiento incorrecto de consultas.
1.2 Alcance
Las disposiciones de este documento aplican a:
 Equipo Comercial.
 
 Ejecutivos de cuenta.
 
 Account Managers.
 
 Soporte Comercial.
 
 Líderes de equipos comerciales.
 
 Gerencia Comercial.
 
 Personal que participe en validaciones de oportunidades.
 
 Áreas especializadas que intervengan en procesos de e

In [7]:
# Validar que todas las páginas tengan texto extraído
for pagina in paginas_texto:
    numero = pagina["pagina"]
    texto = pagina["texto"]

    if texto and texto.strip():
        print(f"Página {numero}: OK - {len(texto)} caracteres")
    else:
        print(f"Página {numero}: SIN TEXTO")


Página 1: OK - 383 caracteres
Página 2: OK - 1036 caracteres
Página 3: OK - 2360 caracteres
Página 4: OK - 2472 caracteres
Página 5: OK - 2248 caracteres
Página 6: OK - 2313 caracteres
Página 7: OK - 2533 caracteres
Página 8: OK - 2215 caracteres
Página 9: OK - 1993 caracteres
Página 10: OK - 1534 caracteres


In [9]:
!pip install -q langchain-text-splitters

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configurar el divisor de texto
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Lista para almacenar todos los chunks
chunks = []

chunk_id = 1

for pagina in paginas_texto:
    fragmentos = text_splitter.split_text(pagina["texto"])

    for fragmento in fragmentos:
        chunks.append({
            "chunk_id": chunk_id,
            "pagina": pagina["pagina"],
            "texto": fragmento
        })

        chunk_id += 1

print(f"Total de chunks creados: {len(chunks)}")

Total de chunks creados: 33


In [12]:
# Mostrar los primeros chunks que provienen de la página 4
chunks_pagina_4 = [
    chunk for chunk in chunks
    if chunk["pagina"] == 4
]

for chunk in chunks_pagina_4[:3]:
    print(f"Chunk ID: {chunk['chunk_id']}")
    print(f"Página: {chunk['pagina']}")
    print("-" * 50)
    print(chunk["texto"])
    print("\n" + "=" * 80 + "\n")

Chunk ID: 8
Página: 4
--------------------------------------------------
NexaShip
NX-POL-001 | Versión 1.0 | Uso Interno
Políticas Internas de NexaShip
Página 4
 fechas de desarrollo de integraciones personalizadas;
 
 modificaciones específicas del producto;
 
 condiciones operativas excepcionales;
 
 coberturas no verificadas;
 
 niveles de servicio no documentados;
 
 resultados dependientes de terceros;
 
 excepciones comerciales pendientes de aprobación.
 
Cuando una solicitud se encuentre en evaluación, deberá comunicarse como sujeta a validación.
2.3 Integraciones estándar y solicitudes personalizadas
Una integración podrá presentarse como estándar únicamente cuando figure expresamente como disponible en la
documentación vigente.
Cuando un prospecto utilice una plataforma desarrollada internamente, un ERP no documentado, una arquitectura


Chunk ID: 9
Página: 4
--------------------------------------------------
documentación vigente.
Cuando un prospecto utilice una plataf

In [13]:
import re

def limpiar_texto(texto):
    # Eliminar caracteres de control extraños
    texto = texto.replace("\x7f", "")

    # Eliminar encabezados y pies repetitivos del PDF
    texto = re.sub(r"^NexaShip\s*$", "", texto, flags=re.MULTILINE)
    texto = re.sub(
        r"^NX-POL-001\s*\|\s*Versión 1\.0\s*\|\s*Uso Interno\s*$",
        "",
        texto,
        flags=re.MULTILINE
    )
    texto = re.sub(
        r"^Políticas Internas de NexaShip\s*$",
        "",
        texto,
        flags=re.MULTILINE
    )
    texto = re.sub(
        r"^Página\s+\d+\s*$",
        "",
        texto,
        flags=re.MULTILINE
    )

    # Reducir exceso de líneas vacías
    texto = re.sub(r"\n\s*\n+", "\n\n", texto)

    return texto.strip()


# Limpiar las páginas extraídas
paginas_limpias = []

for pagina in paginas_texto:
    paginas_limpias.append({
        "pagina": pagina["pagina"],
        "texto": limpiar_texto(pagina["texto"])
    })

print(f"Páginas limpiadas: {len(paginas_limpias)}")

Páginas limpiadas: 10


In [14]:
# Regenerar chunks usando únicamente contenido útil y limpio
chunks_limpios = []

chunk_id = 1

for pagina in paginas_limpias:
    # Excluir portada y página de control/índice
    if pagina["pagina"] <= 2:
        continue

    fragmentos = text_splitter.split_text(pagina["texto"])

    for fragmento in fragmentos:
        chunks_limpios.append({
            "chunk_id": chunk_id,
            "pagina": pagina["pagina"],
            "texto": fragmento
        })

        chunk_id += 1

print(f"Total de chunks limpios creados: {len(chunks_limpios)}")

Total de chunks limpios creados: 33


In [15]:
chunks_pagina_4_limpios = [
    chunk for chunk in chunks_limpios
    if chunk["pagina"] == 4
]

for chunk in chunks_pagina_4_limpios[:2]:
    print(f"Chunk ID: {chunk['chunk_id']}")
    print(f"Página: {chunk['pagina']}")
    print("-" * 50)
    print(chunk["texto"])
    print("\n" + "=" * 80 + "\n")

Chunk ID: 6
Página: 4
--------------------------------------------------
fechas de desarrollo de integraciones personalizadas;

 modificaciones específicas del producto;

 condiciones operativas excepcionales;

 coberturas no verificadas;

 niveles de servicio no documentados;

 resultados dependientes de terceros;

 excepciones comerciales pendientes de aprobación.


Chunk ID: 7
Página: 4
--------------------------------------------------
Cuando una solicitud se encuentre en evaluación, deberá comunicarse como sujeta a validación.
2.3 Integraciones estándar y solicitudes personalizadas
Una integración podrá presentarse como estándar únicamente cuando figure expresamente como disponible en la
documentación vigente.
Cuando un prospecto utilice una plataforma desarrollada internamente, un ERP no documentado, una arquitectura
personalizada, flujos de sincronización especiales o requerimientos técnicos fuera de las capacidades documentadas,
la persona comercial deberá registrar el requerim

In [16]:
!pip install -q sentence-transformers

In [17]:
from sentence_transformers import SentenceTransformer

# Modelo multilingüe para búsqueda semántica
modelo_embeddings = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Extraer únicamente el texto de cada chunk
textos_chunks = [
    chunk["texto"]
    for chunk in chunks_limpios
]

# Convertir los chunks en vectores numéricos
embeddings_chunks = modelo_embeddings.encode(
    textos_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"Total de chunks vectorizados: {len(embeddings_chunks)}")
print(f"Dimensión de cada embedding: {embeddings_chunks.shape[1]}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total de chunks vectorizados: 33
Dimensión de cada embedding: 384


In [18]:
import numpy as np

def buscar_evidencia(pregunta, top_k=3):
    # Convertir la pregunta en un embedding
    embedding_pregunta = modelo_embeddings.encode(
        [pregunta],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Calcular similitud semántica con todos los chunks
    similitudes = embeddings_chunks @ embedding_pregunta

    # Obtener los índices de los chunks más relevantes
    indices_relevantes = np.argsort(similitudes)[::-1][:top_k]

    resultados = []

    for indice in indices_relevantes:
        chunk = chunks_limpios[indice]

        resultados.append({
            "chunk_id": chunk["chunk_id"],
            "pagina": chunk["pagina"],
            "similitud": float(similitudes[indice]),
            "texto": chunk["texto"]
        })

    return resultados

In [19]:
pregunta = "¿Puedo prometerle a un cliente una integración personalizada?"

resultados = buscar_evidencia(pregunta)

print(f"Pregunta: {pregunta}")
print("=" * 80)

for resultado in resultados:
    print(f"Chunk ID: {resultado['chunk_id']}")
    print(f"Página: {resultado['pagina']}")
    print(f"Similitud: {resultado['similitud']:.4f}")
    print("-" * 50)
    print(resultado["texto"])
    print("\n" + "=" * 80 + "\n")

Pregunta: ¿Puedo prometerle a un cliente una integración personalizada?
Chunk ID: 5
Página: 3
Similitud: 0.6006
--------------------------------------------------
exista una funcionalidad similar;

 una persona colaboradora recuerde un caso previo;

 un proveedor externo ofrezca una capacidad equivalente;

 técnicamente parezca posible desarrollarla.

2.2 Compromisos frente a prospectos y clientes
El personal comercial no podrá comprometer, sin autorización correspondiente:
 fechas de lanzamiento de funcionalidades futuras;


Chunk ID: 7
Página: 4
Similitud: 0.5471
--------------------------------------------------
Cuando una solicitud se encuentre en evaluación, deberá comunicarse como sujeta a validación.
2.3 Integraciones estándar y solicitudes personalizadas
Una integración podrá presentarse como estándar únicamente cuando figure expresamente como disponible en la
documentación vigente.
Cuando un prospecto utilice una plataforma desarrollada internamente, un ERP no documentado, una